### Solving eigenvalues of generators using finite difference method

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Goal

Consider the generator 
\begin{equation}
  \mathcal{L} g = - \nabla V \cdot \nabla g + \frac{1}{\beta} \Delta g
\end{equation}
associated to the SDE
\begin{equation}
  dX_t = -\nabla V(X_t)\,dt + \sqrt{2\beta^{-1}} dB_t\,, \quad t > 0\,.
\end{equation}

Our goal is to compute the eigenpairs of $\mathcal{L}$.

We study two potential functions $V$ in 1D.

### Theory

In the previous exercises, we have shown that the operator

$$\widetilde{\mathcal{L}} g = \pi^{\frac{1}{2}} \mathcal{L}\pi^{-\frac{1}{2}} g= \frac{1}{\beta}\Delta g +
		\Big(\frac{1}{2} \Delta V - \frac{\beta}{4} |\nabla V|^2\Big)g$$

is self-adjoint under the standard inner product, where $\pi(x) = \frac{1}{Z} \mathrm{e}^{-\beta V(x)}$. 

In 1D, we have 
$$\widetilde{\mathcal{L}} g= \frac{1}{\beta} g'' +
		\Big(\frac{1}{2}  V'' - \frac{\beta}{4} |V'|^2\Big)g.$$

It is easy to see that 

$$\widetilde{\mathcal{L}}\varphi=\lambda \varphi \quad \Longleftrightarrow \quad \mathcal{L} \pi^{-\frac{1}{2}}\varphi = \lambda \pi^{-\frac{1}{2}}\varphi$$

Therefore, both operators have the same eigenvalues, and the eigenfunctions are related as $\varphi \rightarrow \pi^{-\frac{1}{2}}\varphi$.

In the following, we solve the eigenpairs $(\lambda, \varphi)$ of
$$\widetilde{\mathcal{L}}\varphi=\lambda \varphi$$
using finite difference method, 
and obtain the eigenpairs of $\mathcal{L}$ as $(\lambda, \pi^{-\frac{1}{2}}\varphi)$.

### Finite Difference Method:

**Space truncation**:

Truncate the space $\mathbb{R}$ into the finite interval $[-x_b, x_b]$ and divide it into $n_x$ cells with equal length $d_x = \frac{2x_b}{n_x}$. Let $$x_i=-x_b + (i + 0.5) * d_x, \quad i= 0, 1, \dots, n_x - 1$$ be the cell centers.

**Boundary condition**: 
    on the boundary $x=-x_b, x_b$, we assume Neuman boundary condition for **the eigenfunction of $\mathcal{L}$**.
Based on the discussion above, this means that we assume that $$(\pi^{-\frac{1}{2}} \varphi)'=(\mathrm{e}^{\frac{\beta V}{2}} \varphi)' = 0 \Longleftrightarrow \varphi' + \frac{\beta V'}{2} \varphi=0, \quad \mbox{for}~x=\pm x_b.$$

**Finite Difference for $-\widetilde{\mathcal{L}}$**:

Let $\varphi_i$ be the value of $\varphi$ at center $x_i$.

For $i=1,2,\dots, n_x-1$, we have

\begin{equation}
-\widetilde{\mathcal{L}} \varphi = -\frac{1}{\beta} \varphi'' -
		\Big(\frac{1}{2}  V'' - \frac{\beta}{4} |V'|^2\Big)\varphi \approx \frac{2\varphi_{i} - \varphi_{i+1}-\varphi_{i-1}}{\beta d_x^2} - 
        \Big(\frac{1}{2}  V''(x_i) - \frac{\beta}{4} |V'(x_i)|^2\Big)\varphi_i
        \end{equation}

For $i=0$, we denote by $\varphi_{-1}$ the value of $\varphi$ at $x=-x_b - 0.5 * d_x$. Using the boundary condition, we assume the relation 
$$
\frac{\varphi_{0} - \varphi_{-1}}{d_x} + \frac{\beta V'(x_0)}{2} \varphi_0=0
$$
Using it, we can get rid of $\varphi_{-1}$ and obtain
$$-\widetilde{\mathcal{L}} \varphi_0 \approx \frac{\big(1 - \frac{\beta V'(x_0) d_x}{2}\big)\varphi_{0} - \varphi_{1}}{\beta d_x^2} - 
        \Big(\frac{1}{2}  V''(x_0) - \frac{\beta}{4} |V'(x_0)|^2\Big)\varphi_0
        $$

Similarly, for $i=n_x - 1$, we have
$$-\widetilde{\mathcal{L}} \varphi_{n_x-1} \approx \frac{\big(1 +\frac{\beta V'(x_{n_x-1}) d_x}{2}\big)\varphi_{n_x-1} - \varphi_{n_x-2}}{\beta d_x^2} - 
        \Big(\frac{1}{2}  V''(x_{n_x-1}) - \frac{\beta}{4} |V'(x_{n_x-1})|^2\Big)\varphi_{n_x-1}
        $$

As a result, we obtain a matrix eigenvalue problem.

In [ ]:
def eigen_solver(v, beta, xmax, nx):
    
    # size of cells
    dx = 2 * xmax / nx
    
    mat = np.zeros((nx, nx))
    
    # the operator is discretized into a matrix, according to the expression above.
    
    # non-boundary cells
    for i in range(1,nx-1):   
        # cell center
        x = -xmax + (i+0.5) * dx
        # finite difference for laplacian
        mat[i, i-1] = -1 / (dx**2 * beta)
        mat[i, i+1] = -1 / (dx**2 * beta)
        mat[i, i] = 2 / (dx**2 * beta) 
        # diagonal term
        mat[i,i] -=  0.5 * v.laplacian(x) - 0.25 * beta * v.grad(x)**2
    
    # cell near the left boundary
    x = -xmax + 0.5 * dx
    mat[0, 0] = (1.0 - beta * v.grad(x) * 0.5 * dx) / (dx**2 * beta)
    mat[0, 1] = -1 / (dx**2 * beta)    
    mat[0, 0] -=  0.5 * v.laplacian(x) - 0.25 * beta * v.grad(x)**2
    
    # cell near the right boundary
    x = xmax - 0.5 * dx     
    mat[nx-1, nx-1] = (1.0 + beta * v.grad(x) * 0.5 * dx) / (dx**2 * beta)
    mat[nx-1, nx-2] = -1 / (dx**2 * beta)    
    mat[nx-1,nx-1] -=  0.5 * v.laplacian(x) - 0.25 * beta * v.grad(x)**2
    
    eigs, vecs = np.linalg.eigh(mat)
    
    return eigs, vecs

### Example 1:

quadratic potential $V(x) = \frac{x^2}{2}$.

For this example, the process is OU process and we have computed analytical solutions. 

In [ ]:
class quadratic_v():
    def v(x):
        return x*x / 2
    def grad(x):
        return x
    def laplacian(x):
        return 1.0 

beta = 1.5
# solve the problem on the interval [-xmax, xmax]
xmax = 10.0
# number of cells for finite difference
nx = 4000

eigvals, eigvecs = eigen_solver(quadratic_v, beta=beta, xmax=xmax, nx=nx)

# Hermite polynomials
def H(x, n, beta):
    if n == 0:
        return np.ones(x.shape[0])
    if n == 1:
        return beta * x
    if n == 2:
        return beta**2 * x**2 - beta
    if n == 3:
        return (beta*x)**3 - 3 * beta**2 * x

fig, ax = plt.subplots(1, 2, figsize=(12,5))

n = 100
inds = np.linspace(0, n, n)
ax[0].plot(inds, eigvals[0:n], marker='x', markersize=3, label='eigvals')
ax[0].plot(inds, inds, label='truth')
ax[0].legend()
ax[0].set_title(f"the first {n} eigenvalues")

dx = 2 * xmax / nx
x = np.linspace(-xmax, xmax, nx, endpoint=False) + dx * 0.5
pi_half = np.exp(-0.5 * quadratic_v.v(x) * beta)

for i in range(3):
    # get the eigenfunction of the generator
    vec = eigvecs[:,i] / pi_half
    s = np.sqrt(np.mean(eigvecs[:,i]**2))
    ax[1].plot(x, vec / s, label=f"{i+1}th eigenfunction")
    # analyitcal solution
    hn = H(x, i, beta)
    s = np.sqrt(np.mean(hn**2 * pi_half**2))
    ax[1].plot(x, hn / s, '--', label=f"H{i}")
ax[1].legend()
ax[1].set_title('eigenfunctions')

### Example 2:

double well potential $V(x) = \frac{(x^2-1)^2}{4}$.

In [ ]:
class dw1d_v():
    def v(x):
        return (x*x - 1)**2 / 4
    def grad(x):
        return x * (x**2 - 1)
    def laplacian(x):
        return 3 * x**2 - 1 
        
beta = 4.0
# solve the problem on the interval [-xmax, xmax]
xmax = 3.0
# number of cells for finite difference
nx = 4000

eigvals, eigvecs = eigen_solver(dw1d_v, beta=beta, xmax=xmax, nx=nx)

fig, ax = plt.subplots(1, 2, figsize=(12,5))

n = 10
inds = np.linspace(0, n, n)
ax[0].plot(inds, eigvals[0:n], marker='x', markersize=3, label='eigvals')
ax[0].legend()
ax[0].set_title(f"the first {n} eigenvalues")

dx = 2 * xmax / nx
x = np.linspace(-xmax, xmax, nx, endpoint=False) + dx * 0.5
pi_half = np.exp(-0.5 * dw1d_v.v(x) * beta)

for i in range(3):
    # compute the eigenfunction of the generator
    vec = eigvecs[:,i] / pi_half
    s = np.sqrt(np.mean(eigvecs[:,i]**2))
    ax[1].plot(x, vec / s, label=f"{i+1}th eigenfunction")
ax[1].legend()
ax[1].set_title('eigenfunctions')